# BDIViz User Test Case 1

## Supplimentary materials

**GDC Metadata Validation Services:**
https://docs.gdc.cancer.gov/Data_Dictionary/gdcmvs/  
  
**Data Dictionary Viewer:**
https://docs.gdc.cancer.gov/Data_Dictionary/viewer/


---
  
#### Paper info: [link](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9929250/)

_**Multilevel proteomic analyses reveal molecular diversity between diffuse-type and intestinal-type gastric cancer**_  

_Shi W et. al_

Diffuse-type gastric cancer (DGC) and intestinal-type gastric cancer (IGC) are the major histological types of gastric cancer (GC). The molecular mechanism underlying DGC and IGC differences are poorly understood. In this research, we carry out multilevel proteomic analyses, including proteome, phospho-proteome, and transcription factor (TF) activity profiles, of 196 cases covering DGC and IGC in Chinese patients. Integrative proteogenomic analysis reveals ARIDIA mutation associated with opposite prognostic effects between DGC and IGC, via diverse influences on their corresponding proteomes. Systematical comparison and consensus clustering analysis identify three subtypes of DGC and IGC, respectively, based on distinct patterns of the cell cycle, extracellular matrix organization, and immune response-related proteins expression. TF activity-based subtypes demonstrate that the disease progressions of DGC and IGC were regulated by SWI/SNF and NFKB complexes. Furthermore, inferred immune cell infiltration and immune clustering show Th1/Th2 ratio is an indicator for immunotherapeutic effectiveness, which is validated in an independent GC anti-PD1 therapeutic patient group. Our multilevel proteomic analyses enable a more comprehensive understanding of GC and can further advance the precision medicine.


## 0. Load Dataset

In [1]:
import pandas as pd
import numpy as np

import panel as pn
import bdikit as bdi
from bdikit.visualization.schema_matching import BDISchemaMatchingHeatMap

In [3]:
# Li GX et al.
Li = pd.read_csv("Li_GX_et_al_cleaned.csv").drop(columns=["Unnamed: 0"]).infer_objects()

# Shi W et al.
Shi = (
    pd.read_csv("Shi_W_et_al_cleaned.csv").drop(columns=["Unnamed: 0"]).infer_objects()
)

source = Shi
target = "gdc"

In [4]:
source

,"Lauren subtype (D, diffused type; I, intestinal type; M,mixed type)",Gender,Age,Stage,"Live status (1=dead, 0=alive)",Overall survival (days),"Progression (1=progressed, 0=not progressed)",Disease free survival (days),Tumor_location,"Lymphovascular invasion (0=negative, 1=positive)",GeneSymbol,Mutation
0,D,male,80,IV,1,87,1,45,Antrum,1,ARID1A,Nonsense_Mutation
1,D,male,80,IV,1,87,1,45,Antrum,1,DNAH7,Missense_Mutation
2,D,male,80,IV,1,87,1,45,Antrum,1,GLI3,Missense_Mutation
3,D,male,80,IV,1,87,1,45,Antrum,1,HMBOX1,Missense_Mutation
4,D,male,80,IV,1,87,1,45,Antrum,1,TP53,Missense_Mutation
...,...,...,...,...,...,...,...,...,...,...,...,...
95,D,male,56,IIIB,0,1204,0,1113,Cardia,1,FAT4,Missense_Mutation
96,D,male,56,IIIB,0,1204,0,1113,Cardia,1,BRAF,Missense_Mutation
97,D,male,56,IIIB,0,1204,0,1113,Cardia,1,FRMD4A,Missense_Mutation
98,D,male,56,IIIB,0,1204,0,1113,Cardia,1,ATM,Missense_Mutation


## 1. Visualization Interface

In [5]:
heatmap_manager = BDISchemaMatchingHeatMap(
    source,
    target=target,
    top_k=40,
    # additional_sources=additional_sources,
    ai_assitant=True,
)
heatmap_manager.plot_heatmap()

Extracting features from 12 columns...


  0%|          | 0/12 [00:00<?, ?it/s]

Table features loaded for 734 columns


Column(scroll=True)
    [0] Row(styles={'background': '...}, width=1200)
        [0] Select(name='Source Column', options=['Lauren subtype (D, ...], value='Lauren subtype (D, ..., width=120)
        [1] Select(name='Candidate Type', options=['All', 'enum', ...], value='All', width=120)
        [2] IntSlider(end=5, name='Similar Sources', width=150)
        [3] FloatSlider(name='Candidate Threshold', step=0.01, value=0.1, width=150)
        [4] Column
            [0] Button(button_type='success', name='Accept Match')
            [1] Button(button_type='danger', name='Reject Match')
            [2] Button(button_type='warning', name='Discard Column')
        [5] Column
            [0] Button(button_style='outline', button_type='warning', name='Undo')
            [1] Button(button_style='outline', button_type='primary', name='Redo')
        [6] Column
            [0] Button(button_type='primary', name='Show/Hide AI Assistant')
            [1] Button(button_type='primary', name='Show/Hide Operation Log')
    [1] ParamFunction(function, _pane=FloatPanel, defer_load=False)
    [2] Spacer(height=5)
    [3] Column
        [0] ParamFunction(function, _pane=Column, defer_load=False)

## 2. Schema Matching with BDIViz Results

In [12]:
from bdikit.mapping_algorithms.column_mapping.algorithms import TwoPhaseSchemaMatcher

two_phase_viz = TwoPhaseSchemaMatcher(top_k_matcher=heatmap_manager)
column_mappings = bdi.match_schema(source, target=target, method=two_phase_viz)
column_mappings

,source,target
0,"Lauren subtype (D, diffused type; I, intestina...",tumor_shape
1,Gender,gender
2,Age,age_at_diagnosis
3,Stage,ajcc_pathologic_stage
4,"Live status (1=dead, 0=alive)",vital_status
5,Overall survival (days),days_to_death
6,"Progression (1=progressed, 0=not progressed)",progression_or_recurrence
7,Disease free survival (days),days_to_last_known_disease_status
8,Tumor_location,site_of_resection_or_biopsy
9,"Lymphovascular invasion (0=negative, 1=positive)",lymphatic_invasion_present


## 3. Value Matchings

In [ ]:
column_mappings = column_mappings[column_mappings["target"].str.strip().astype(bool)]
mappings = bdi.match_values(
    source,
    column_mapping=column_mappings,
    target=target,
    method="tfidf",
)

for mapping in mappings:
    print(f"{mapping.attrs['source']} => {mapping.attrs['target']}")
    display(mapping)
    print("")